# Notebook 4: Build OpenCLIP Dataset (Case-Level Train/Val/Test Split)

# Load Master Dataset

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import json
from pathlib import Path

import pandas as pd
from sklearn.model_selection import train_test_split

CONFIG = Path("/content/drive/MyDrive/Surgical-VLM/configs/config.json")
with open(CONFIG) as f:
    config = json.load(f)

PROCESSED_DIR = Path(config["processed_dir"])

master_df = pd.read_parquet(PROCESSED_DIR / "master_dataset.parquet")
print(master_df.shape)
print(master_df["task"].value_counts())


(661962, 16)
task
Triplet Recognition             154214
Safety Assessment               118819
Instrument Recognition          111724
Action Recognition              105642
Phase Recognition                75082
Surgical Image Captioning        62134
Tissue and Organ Recognition     34347
Name: count, dtype: int64


# Basic Cleaning

In [ ]:
# Drop rows with no answer text (CLIP needs a text target for every row)
openclip_df = master_df[master_df["answer"].notna() & (master_df["answer"].str.strip() != "")].copy()

print(openclip_df.shape, "after dropping empty answers")


(661962, 16) after dropping empty answers


# Case-Level Train/Val/Test Split

Splitting by case_id (video), not by row, so no two frames from the same video end up in different splits — CholecT50 frames within a video are highly correlated (near-duplicate consecutive frames, same anatomy/lighting), and a row-level split would leak.

In [ ]:
unique_cases = sorted(openclip_df["case_id"].unique())
print(f"{len(unique_cases)} unique cases")

train_cases, temp_cases = train_test_split(unique_cases, test_size=0.3, random_state=42)
val_cases, test_cases = train_test_split(temp_cases, test_size=0.5, random_state=42)

train_df = openclip_df[openclip_df["case_id"].isin(train_cases)].reset_index(drop=True)
val_df = openclip_df[openclip_df["case_id"].isin(val_cases)].reset_index(drop=True)
test_df = openclip_df[openclip_df["case_id"].isin(test_cases)].reset_index(drop=True)

print("Train:", train_df.shape, f"({len(train_cases)} cases)")
print("Val  :", val_df.shape, f"({len(val_cases)} cases)")
print("Test :", test_df.shape, f"({len(test_cases)} cases)")

# Verify no leakage
assert set(train_cases).isdisjoint(val_cases)
assert set(train_cases).isdisjoint(test_cases)
assert set(val_cases).isdisjoint(test_cases)
print("\n No case leakage across splits.")


43 unique cases
Train: (505527, 16) (30 cases)
Val  : (50986, 16) (6 cases)
Test : (105449, 16) (7 cases)

✓ No case leakage across splits.


# Build `text` Column

In [ ]:
for df in [train_df, val_df, test_df]:
    df["text"] = df["answer"]

print(train_df[["task", "question", "answer", "text"]].head(3))


                 task                                           question  \
0  Action Recognition  Given the laparoscopic cholecystectomy image, ...   
1  Action Recognition  Given the laparoscopic cholecystectomy image, ...   
2  Action Recognition  Given the laparoscopic cholecystectomy image, ...   

            answer             text  
0  grasp, aspirate  grasp, aspirate  
1            grasp            grasp  
2         aspirate         aspirate  


# Deduplicate

In [ ]:
for name, df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    before = len(df)
    df.drop_duplicates(subset=["image_id", "question", "answer"], inplace=True)
    df.reset_index(drop=True, inplace=True)
    print(f"{name}: {before} -> {len(df)} after dedup")


train: 505527 -> 505527 after dedup
val: 50986 -> 50986 after dedup
test: 105449 -> 105449 after dedup


# Save

In [ ]:
train_df.to_parquet(PROCESSED_DIR / "openclip_train.parquet", index=False)
val_df.to_parquet(PROCESSED_DIR / "openclip_val.parquet", index=False)
test_df.to_parquet(PROCESSED_DIR / "openclip_test.parquet", index=False)

print("Saved:")
print(PROCESSED_DIR / "openclip_train.parquet", train_df.shape)
print(PROCESSED_DIR / "openclip_val.parquet", val_df.shape)
print(PROCESSED_DIR / "openclip_test.parquet", test_df.shape)


Saved:
/content/drive/MyDrive/Surgical-VLM/processed/openclip_train.parquet (505527, 17)
/content/drive/MyDrive/Surgical-VLM/processed/openclip_val.parquet (50986, 17)
/content/drive/MyDrive/Surgical-VLM/processed/openclip_test.parquet (105449, 17)
